# T2.4 – Qualitative Analysis: Config 1 vs Config 3

Compares detections from the baseline (Config 1) and best (Config 3) models against ground truth on a sample of validation images.

## 1. Setup & Imports

In [ ]:
# Install YOLO
!pip install -q -U ultralytics

# Imports
import cv2
import glob
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO


## 2. Load & Extract the Dataset

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Unzip the kitti3k_yolo dataset (needed for images and validation labels)
!unzip -q "/content/drive/MyDrive/kitti3k_yolo.zip"
!mv /content/kitti3k_yolo/kitti3k_yolo/* /content/kitti3k_yolo/ 2>/dev/null || true

print("Setup complete!")


## 3. IoU Matching & Side-by-Side Visual Comparison

In [ ]:
# --- Configuration ---
CONF_THRESH = 0.50       # Confidence threshold
MATCH_IOU_THRESH = 0.50  # IoU matching threshold
image_ids = ['000136', '000232', '000189']
dataset_path = "/content/kitti3k_yolo"

# Load saved weight files directly from Google Drive
model_cfg1_path = "/content/drive/MyDrive/cfg1_best.pt"
model_cfg3_path = "/content/drive/MyDrive/cfg3_best.pt"


def get_iou(boxA, boxB):
    """Compute the Intersection-over-Union of two [x1, y1, x2, y2] boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea + 1e-6)


def load_gt_boxes(label_path, img_w, img_h):
    """Load YOLO-format ground-truth boxes and convert them to pixel [x1, y1, x2, y2]."""
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls_id, x_c, y_c, w, h = map(float, parts)
                x1 = (x_c - w/2) * img_w
                y1 = (y_c - h/2) * img_h
                x2 = (x_c + w/2) * img_w
                y2 = (y_c + h/2) * img_h
                boxes.append([x1, y1, x2, y2])
    return boxes


def plot_image(ax, img_path, label_path, model):
    """Run inference on one image and plot TP (green), FP (red), FN (amber) and GT (blue dashed) boxes."""
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_h, img_w, _ = img.shape

    gt_boxes = load_gt_boxes(label_path, img_w, img_h)
    results = model(img_path, conf=CONF_THRESH, verbose=False)[0]
    pred_boxes = results.boxes.xyxy.cpu().numpy()

    matched_gt = set()
    matched_pred = set()

    # Match predictions to ground truth
    for p_idx, p_box in enumerate(pred_boxes):
        best_iou = 0
        best_g_idx = -1
        for g_idx, g_box in enumerate(gt_boxes):
            if g_idx in matched_gt:
                continue
            iou = get_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_g_idx = g_idx

        if best_iou >= MATCH_IOU_THRESH:
            matched_gt.add(best_g_idx)
            matched_pred.add(p_idx)
            # True positive: green
            rect = patches.Rectangle((p_box[0], p_box[1]), p_box[2]-p_box[0], p_box[3]-p_box[1],
                                     linewidth=2, edgecolor='green', facecolor='none')
            ax.add_patch(rect)
        else:
            # False positive: red
            rect = patches.Rectangle((p_box[0], p_box[1]), p_box[2]-p_box[0], p_box[3]-p_box[1],
                                     linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)

    # Plot ground truth & false negatives
    for g_idx, g_box in enumerate(gt_boxes):
        # Ground truth: blue (dashed)
        rect = patches.Rectangle((g_box[0], g_box[1]), g_box[2]-g_box[0], g_box[3]-g_box[1],
                                 linewidth=1, edgecolor='blue', facecolor='none', linestyle='--')
        ax.add_patch(rect)

        if g_idx not in matched_gt:
            # False negative: amber/orange
            rect = patches.Rectangle((g_box[0], g_box[1]), g_box[2]-g_box[0], g_box[3]-g_box[1],
                                     linewidth=2.5, edgecolor='orange', facecolor='none')
            ax.add_patch(rect)

    ax.imshow(img_rgb)
    ax.axis('off')


# Load models from Google Drive
print("Loading models from Google Drive...")
model1 = YOLO(model_cfg1_path)
model3 = YOLO(model_cfg3_path)

# Plot the comparison grid: one row per image, Config 1 vs Config 3
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
fig.suptitle('Qualitative Analysis: Config 1 vs Config 3\nBlue=GT | Green=TP | Red=FP | Amber=FN', fontsize=16)
axes[0, 0].set_title('Configuration 1 (Baseline)')
axes[0, 1].set_title('Configuration 3 (Best)')

for i, img_id in enumerate(image_ids):
    img_search = glob.glob(f"{dataset_path}/images/val/{img_id}.*")
    img_path = img_search[0] if img_search else None
    label_path = f"{dataset_path}/labels/val/{img_id}.txt"

    if img_path and os.path.exists(label_path):
        plot_image(axes[i, 0], img_path, label_path, model1)
        plot_image(axes[i, 1], img_path, label_path, model3)
    else:
        print(f"Could not find files for image ID: {img_id}")

plt.tight_layout()
plt.savefig("qualitative_analysis_t24.png", dpi=300)
plt.show()
